# Fine-tuning BEATs trên Tín hiệu Nhiễu Ước lượng qua FRCRN (Mixture-36l)

Pipeline chuẩn theo **Baseline Đợt 4 (`run1_baseline`)**:
1. **Tín hiệu đầu vào:** Hỗn hợp âm thanh giọng nói + nhiễu `mixture-36l` (16,000 Hz, 4.0s).
2. **Tách nhiễu bằng FRCRN:** Mô hình FRCRN đã tiền huấn luyện (Complex UNet Masking) được đóng băng để ước lượng tín hiệu nhiễu $\hat{\mathbf{n}} = \mathbf{x}_{\text{mix}} - \hat{\mathbf{s}}$.
3. **Chuẩn hóa RMS:** Đưa năng lượng của $\hat{\mathbf{n}}$ về phân phối AudioSet ($\sigma_0 = 0.07925$).
4. **Fine-tuning BEATs:** Backbone ViT-Base (90.34M tham số) với head phân loại 36 lớp được tối ưu hóa bằng AdamW (LR 1e-4, layer decay 0.3, tích lũy gradient 2 bước = effective batch size 32).
5. **Đánh giá toàn diện:** 9 chỉ số phân loại, phân tích theo 6 mức SNR (-5 đến 20 dB), và đánh giá tập Unseen (3,064 mẫu sạch).

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "rules.MD").is_file())
PACKAGE_ROOT = ROOT / "imp" / "ft-beat"
os.chdir(ROOT)
for path in (ROOT, PACKAGE_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print(f"Workspace ROOT: {ROOT}")
print(f"Package ROOT:   {PACKAGE_ROOT}")

In [ ]:
from ft_beat.config import load_config
from ft_beat.runtime import resolve_device

CONFIG_PATH = PACKAGE_ROOT / "config" / "ft_beat.yaml"
config = load_config(CONFIG_PATH)
device = resolve_device()
print(f"Device:               {device}")
print(f"Model Config:         {config.model_config}")
print(f"FRCRN Checkpoint:     {config.frcrn.checkpoint}")
print(f"Effective Batch Size: {config.train.batch_size * config.train.accumulation_steps}")
print(f"Target RMS:           {config.frcrn.target_rms}")

In [ ]:
import logging

from ft_beat.pipeline import load_classifier, load_separator

logger = logging.getLogger("notebook")
logging.basicConfig(level=logging.INFO)

separator = load_separator(config, device, logger)
beats = load_classifier(config, device, logger)

frcrn_params = f"{separator.num_parameters() / 1e6:.2f}M" if separator is not None else "Bypassed (using cached n-hat)"
print(f"FRCRN Frozen Params:    {frcrn_params}")
print(f"BEATs Trainable Params: {beats.num_parameters(trainable_only=True) / 1e6:.2f}M")

In [ ]:
# Chạy huấn luyện theo cấu hình benchmark Đợt 4 Baseline
from ft_beat.pipeline import run_pipeline

RUN_FULL_TRAIN = True  # Đã cấu hình chạy huấn luyện thật

result = run_pipeline(config, smoke=not RUN_FULL_TRAIN, device=device)
print("Hoàn thành pipeline!")
print("Validation Top-1 Acc:", result["val_metrics"].get("top1_accuracy"))
print("Validation Macro-F1: ", result["val_metrics"].get("macro_f1"))